In [1]:
!pip install peft transformers datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 23.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 35.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 18.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 26.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 19.9 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl 

In [12]:
# Step 1: Setup and Installation
!pip install -q peft transformers datasets
from transformers import AutoModelForSequenceClassification, AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType
import torch
from torch.nn.functional import pad
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm

# Step 2: Define Configuration
device = "cuda"
model_name_or_path = 'bert-base-multilingual-uncased'
tokenizer_name_or_path = 'bert-base-multilingual-uncased'
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,  # Correct task type for sequence classification
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="What is the sentiment of this text?",  # Changed to a more specific prompt
    tokenizer_name_or_path=tokenizer_name_or_path,
)
dataset_name = "twitter_complaints"
checkpoint_name = f"{dataset_name}_{model_name_or_path}_{peft_config.peft_type}_{peft_config.task_type}_v1.pt".replace("/", "_")
text_column = "Tweet text"
label_column = "text_label"
max_length = 128  # Increased max_length to accommodate longer texts
lr = 3e-2
num_epochs = 2
batch_size = 8

# Step 3: Load Dataset

dataset = load_dataset("ought/raft", dataset_name)
classes = [k.replace("_", " ") for k in dataset["train"].features["Label"].names]
dataset = dataset.map(lambda x: {"text_label": [classes[label] for label in x["Label"]]}, batched=True, num_proc=1)

# Step 4: Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
target_max_length = max([len(tokenizer(class_label)["input_ids"]) for class_label in classes])
print("Target max length:", target_max_length)

# Step 5: Preprocess Function
def preprocess_function(examples):
    batch_size = len(examples[text_column])
    inputs = [f"{text_column} : {x} Label : " for x in examples[text_column]]
    targets = [str(x) for x in examples[label_column]]
    model_inputs = tokenizer(inputs, truncation=True, max_length=max_length, padding="max_length")
    labels = tokenizer(targets, truncation=True, max_length=target_max_length, padding="max_length")
    labels_tensor = torch.tensor(labels["input_ids"])  # Convert the list to a tensor
    labels = torch.argmax(labels_tensor, dim=1)  # Get the index of the maximum value in each row
    model_inputs["labels"] = labels  # Assign the labels tensor directly
    return model_inputs

# Step 6: Apply Preprocessing
processed_datasets = dataset.map(preprocess_function, batched=True, num_proc=1, remove_columns=dataset["train"].column_names, load_from_cache_file=False, desc="Running tokenizer on dataset")
train_dataset = processed_datasets["train"]
eval_dataset = processed_datasets["test"]

# Step 7: DataLoaders
train_dataloader = DataLoader(train_dataset, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)
eval_dataloader = DataLoader(eval_dataset, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)

# Step 8: Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=len(classes))
model = get_peft_model(model, peft_config)
print(model.print_trainable_parameters())

# Step 9: Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=0, num_training_steps=(len(train_dataloader) * num_epochs))

# Step 10: Training Loop
model = model.to(device)
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        if all(tensor.dim() == 0 for tensor in batch["labels"]):  # Check if all tensors are 0-dimensional
            batch["labels"] = batch["labels"]  # Use the original list of 0-dimensional tensors
        else:
            max_target_length = max(tensor.size(0) for tensor in batch["labels"] if tensor.dim() > 0)  # Get the max length of the tensors
            padded_targets = [pad(tensor, (0, max_target_length - tensor.size(0))) if tensor.dim() > 0 else tensor.unsqueeze(0) for tensor in batch["labels"]]
            batch["labels"] = torch.stack(padded_targets)
        print("Input shape:", batch["input_ids"].shape)
        print("Target shape:", batch["labels"].shape)
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(torch.argmax(outputs.logits, -1).detach().cpu().numpy())

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"Epoch {epoch}: train_ppl={train_ppl}, train_loss={train_epoch_loss}, eval_ppl={eval_ppl}, eval_loss={eval_epoch_loss}")


Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 3108, in _dep_map
    return self.__dep_map
  File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 2901, in __getattr__
    raise AttributeError(attr)
AttributeError: _DistInfoDistribution__dep_map

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 169, in exc_logging_wrapper
    status = run_func(*args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/req_command.py", line 242, in wrapper
    return func(self, options, args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/commands/install.py", line 441, in run
    conflicts = self._determine_conflicts(to_install)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/commands/install.py", line 

KeyboardInterrupt: 

In [2]:
from transformers import AutoModelForSequenceClassification, AutoModelWithLMHead, AutoModelForCausalLM, AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup, BertConfig
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType, PeftType
import torch
from datasets import load_dataset , Dataset
import os
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd

In [3]:

model_name_or_path = 'bert-base-multilingual-uncased'
tokenizer_name_or_path = 'bert-base-multilingual-uncased'
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="Classify if the tweet is a complaint or not:",
    tokenizer_name_or_path=model_name_or_path,
)

In [5]:

dataset_name = "twitter_complaints"
checkpoint_name = f"{dataset_name}_{model_name_or_path}_{peft_config.peft_type}_{peft_config.task_type}_v1.pt".replace(
    "/", "_"
)
text_column = "Tweet text"
label_column = "text_label"
max_length = 64
lr = 3e-2
num_epochs = 1
batch_size = 8

# dataset = pd.read_csv()

# dataset.iloc[0]

column_names = [label_column,"blank" ,text_column]

# Load the dataset without headers and assign the column names
dataset = pd.read_csv("/content/societal_2l_train_hi.csv", header=None, names=column_names).head(500) #examples in dataset
dataset.drop("blank", axis=1, inplace=True)
dataset[label_column] = dataset[label_column].map({0:"negative", 4:"positive"})


In [6]:
print(dataset.head())
# dataset.columns = [label_column, text_column]

  text_label                                         Tweet text
0   negative  @sardanarohit @sudhirchaudhary @arnabgoswamias...
1   positive  @narendramodi प्रिय पीएम, युवाओं को हम #Cashle...
2   negative  @Narendramodi सर पठानकोट निगम AMRIT CITY SCHEO...
3   positive  #jobs #jobsearch ##parliament GST बिल पास करता...
4   positive  @Upsubramanya ppl praise manmohan 4 1991 सुधार...


In [7]:
from sklearn.model_selection import train_test_split

# Split the preprocessed dataset into training and testing sets (80% train, 20% test)
train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)

# Now you have train_dataset and test_dataset for training and testing respectively
print(train_dataset)

    text_label                                         Tweet text
249   positive  #GST को अच्छी तरह से समझें और इसे अपने निर्वाच...
433   positive  @YADAVAKHILESH आप असली SAMAJWADI हैं। ऐसे व्यक...
19    negative  यदि पाक के लोग आतंकवाद के खिलाफ खड़े होने के ल...
322   negative  Pathankot के दागी SP SALWINDER ने @Timesofindi...
332   negative  @Pmoindia, @finminindia @narendramodi केंद्र स...
..         ...                                                ...
106   negative  RT @kumar_ke5hav: #surgicalStrike और एक कमजोर ...
270   negative  @airtelindia @ideacellular @vodafonein @aircel...
348   positive  सर, # #पर प्रतिबंध उसके लिए एक मदद है, जितना अ...
435   positive  GST टीम इंडिया द्वारा एक महान कदम है, परिवर्तन...
102   positive  @narendramodi @pmoindia यह भारत में बदलाव करने...

[400 rows x 2 columns]


In [8]:
dataset_train = Dataset.from_pandas(train_dataset)

In [9]:
dataset_test = Dataset.from_pandas(test_dataset)

In [11]:
# Step 4: Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
# Extract unique classes from the label column
classes = dataset[label_column].unique().tolist()  # Assuming 'label_column' is defined

target_max_length = max([len(tokenizer(class_label)["input_ids"]) for class_label in classes])
print("Target max length:", target_max_length)
# Step 5: Preprocess Function
def preprocess_function(examples):
    batch_size = len(examples[text_column])
    inputs = [f"{text_column} : {x} Label : " for x in examples[text_column]]
    targets = [str(x) for x in examples[label_column]]
    model_inputs = tokenizer(inputs, truncation=True, max_length=max_length, padding="max_length")
    labels = tokenizer(targets, truncation=True, max_length=target_max_length, padding="max_length")
    labels_tensor = torch.tensor(labels["input_ids"])  # Convert the list to a tensor
    labels = torch.argmax(labels_tensor, dim=1)  # Get the index of the maximum value in each row
    model_inputs["labels"] = labels  # Assign the labels tensor directly
    return model_inputs

# Step 6: Apply Preprocessing
processed_datasets = dataset.map(preprocess_function, batched=True, num_proc=1, remove_columns=dataset["train"].column_names, load_from_cache_file=False, desc="Running tokenizer on dataset")
train_dataset = processed_datasets["train"]
eval_dataset = processed_datasets["test"]

# Step 7: DataLoaders
train_dataloader = DataLoader(train_dataset, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)
eval_dataloader = DataLoader(eval_dataset, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)

# Step 8: Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=len(classes))
model = get_peft_model(model, peft_config)
print(model.print_trainable_parameters())

# Step 9: Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=0, num_training_steps=(len(train_dataloader) * num_epochs))

# Step 10: Training Loop
model = model.to(device)
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        if all(tensor.dim() == 0 for tensor in batch["labels"]):  # Check if all tensors are 0-dimensional
            batch["labels"] = batch["labels"]  # Use the original list of 0-dimensional tensors
        else:
            max_target_length = max(tensor.size(0) for tensor in batch["labels"] if tensor.dim() > 0)  # Get the max length of the tensors
            padded_targets = [pad(tensor, (0, max_target_length - tensor.size(0))) if tensor.dim() > 0 else tensor.unsqueeze(0) for tensor in batch["labels"]]
            batch["labels"] = torch.stack(padded_targets)
        print("Input shape:", batch["input_ids"].shape)
        print("Target shape:", batch["labels"].shape)
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(torch.argmax(outputs.logits, -1).detach().cpu().numpy())

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"Epoch {epoch}: train_ppl={train_ppl}, train_loss={train_epoch_loss}, eval_ppl={eval_ppl}, eval_loss={eval_epoch_loss}")


Target max length: 3


AttributeError: 'DataFrame' object has no attribute 'map'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def preprocess_function(examples):
    batch_size = len(examples[text_column])
    inputs = [f"{text_column} : {x} Label : " for x in examples[text_column]]

    targets = [str(x) for x in examples[label_column]]

    model_inputs = tokenizer(inputs)
    labels = tokenizer(targets)
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i] + [tokenizer.pad_token_id]
        # print(i, sample_input_ids, label_input_ids)
        model_inputs["input_ids"][i] = sample_input_ids + label_input_ids
        labels["input_ids"][i] = [-100] * len(sample_input_ids) + label_input_ids
        model_inputs["attention_mask"][i] = [1] * len(model_inputs["input_ids"][i])
    # print(model_inputs)
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i]
        model_inputs["input_ids"][i] = [tokenizer.pad_token_id] * (
            max_length - len(sample_input_ids)
        ) + sample_input_ids
        model_inputs["attention_mask"][i] = [0] * (max_length - len(sample_input_ids)) + model_inputs[
            "attention_mask"
        ][i]
        labels["input_ids"][i] = [-100] * (max_length - len(sample_input_ids)) + label_input_ids
        model_inputs["input_ids"][i] = torch.tensor(model_inputs["input_ids"][i][:max_length])
        model_inputs["attention_mask"][i] = torch.tensor(model_inputs["attention_mask"][i][:max_length])
        labels["input_ids"][i] = torch.tensor(labels["input_ids"][i][:max_length])
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
processed_datasets_train = dataset_train.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_train.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)

In [ ]:
processed_datasets_test = dataset_test.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_test.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)

In [ ]:
# train_dataset = train_dataset
# eval_dataset = test_dataset

train_dataloader = DataLoader(
    processed_datasets_train, shuffle=True, collate_fn=default_data_collator, batch_size=8, pin_memory=True
)
# eval_dataloader = DataLoader(eval_dataset, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)

In [ ]:
test_dataloader = DataLoader(
    processed_datasets_test, shuffle=True, collate_fn=default_data_collator, batch_size=8, pin_memory=True
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name_or_path)
model = get_peft_model(model, peft_config)
print(model.print_trainable_parameters())

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

In [ ]:
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

In [ ]:
inputs = tokenizer(
    f'{text_column} : {"@nationalgridus This is not good"} Label : ',
    return_tensors="pt",
)

In [ ]:
model.to(device)

with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(
        input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], max_new_tokens=10, eos_token_id=3
    )
    print(tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True))

['Tweet text : @nationalgridus This is not good Label : no complaint chiesto parlanoExtrait useHistory useHistory chied dicono']


In [ ]:
torch.save(model, 'navarasa_model.pth')